Dashboard for Government Revenue

=================================

Section 1
Where does federal government revenue come from?
Dropdown: Level (RM), % of Revenue, % of Revenue Group

Revenue by Source (stacked bar chart, full-width): DONE

    Direct Taxes (blue), Non-Tax Revenue (grey), Non-Revenue Receipts (orange), Indirect Taxes (red), 


Direct Tax Revenue (stacked bar chart, half-width): 

    Corporates (blue), Petroleum (grey), Others (Orange), Individuals (Red)


Indirect Tax Revenue (stacked bar chart, half-width): 

    Sales Tax (blue), Duties (grey), Others (Orange), Services Tax (Red)

=================================

Section 2

Key revenue sufficiency metrics
Pills: Annual, Quarterly

Timeseries:

Revenue as % of Nominal GDP (Gov Revenue / Nominal GDP) - Quarter & Annual

Government Deficit (Gov Revenue - Gov Expenditure) - Quarter & Annual

Revenue per Capita (Gov Revenue / Total Population) - Quarter & Annual (Using only annual population)

=================================

Section 3

A breakdown of direct tax revenue
Pills: Annual, Quarterly
Dropdown: Level (RM), YoY Growth

Timeseries: Corporate Income Tax, Individual Income Tax, Petroleum Income Tax, Stamp Duties, Others

=================================

Section 4

A breakdown of indirect tax revenue
Pills: Annual, Quarterly
Dropdown: Level (RM), YoY Growth

Timeseries: Sales Tax, Services Tax, Export Duties, Import Duties, Excise Duties, Others

In [2]:
import pandas as pd
import numpy as np
import json
import duckdb
from datetime import datetime, timedelta
from helper import write_csv_parquet, derive_df_with_growth, get_recession_dates, get_cols_mhs

# ==========================================
# 1. DATA FETCHING
# ==========================================
pop_url = "https://storage.dosm.gov.my/population/population_malaysia.parquet"
gdp_annual_url = "https://storage.dosm.gov.my/gdp/gdp_gni_annual_nominal.parquet"
gdp_quarter_url = "https://storage.dosm.gov.my/gdp/gdp_qtr_nominal.parquet"

con = duckdb.connect()
population = con.execute(f"SELECT date, population * 1000 AS population FROM '{pop_url}' WHERE sex = 'both' AND age = 'overall' AND ethnicity = 'overall'").df()
gdp = con.execute(f"SELECT date, gdp * 1000000 AS gdp, 'Annual' AS freq FROM '{gdp_annual_url}' WHERE series = 'abs' UNION ALL SELECT date, value * 1000000 AS gdp, 'Quarterly' AS freq FROM '{gdp_quarter_url}' WHERE series = 'abs'").df()
con.close()

def process_3_1_data(filename="3.1.1.xlsx", name="gov_revenue", rows=9):
    df = pd.read_excel(filename, skiprows=rows)
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    df = df.dropna(how='all')
    cols = get_cols_mhs(db=name, status="initial")
    df.columns = cols

    df['year'] = df['year'].astype(str).str[:4]
    df['year'] = pd.to_numeric(df['year'], errors='coerce').ffill()
    df['freq'] = np.where(df['quarter'].str.contains('Q', na=False), 'Quarterly', 'Annual')
    
    df['year'] = df['year'].ffill()
    quarter_map = {'1': '01', '2': '04', '3': '07', '4': '10'}
    df['quarter'] = df['quarter'].fillna("1Q")
    
    df['temp_date_str'] = (
        df['year'].astype(int).astype(str) + '-' + 
        df['quarter'].str[0].map(quarter_map) + '-01'
    )
    df['date'] = pd.to_datetime(df['temp_date_str'])
    df = df.drop(columns=['temp_date_str', 'year', 'quarter'])

    numeric_cols = df.select_dtypes(include=['number']).columns
    cols_to_fix = [c for c in numeric_cols if c not in ['date', 'freq']]
    df[cols_to_fix] = df[cols_to_fix] * 1_000_000
    return df

gov_revenue = process_3_1_data("3.1.1.xlsx", "gov_revenue", 9)
gov_opex = process_3_1_data("3.1.2.xlsx", "gov_opex", 6)
gov_devex = process_3_1_data("3.1.3.xlsx", "gov_devex", 6)

con = duckdb.connect()
df = con.execute("""
    SELECT
        gov_revenue.*,
        'timeseries' AS type,
        gov_revenue.total - gov_opex.total - gov_devex.total AS gov_deficit,
        gov_revenue.total / gdp.gdp * 100 AS revenue_per_gdp
    FROM gov_revenue
    LEFT JOIN gov_opex ON gov_revenue.date = gov_opex.date AND gov_revenue.freq = gov_opex.freq
    LEFT JOIN gov_devex ON gov_revenue.date = gov_devex.date AND gov_revenue.freq = gov_devex.freq
    LEFT JOIN gdp ON gov_revenue.date = gdp.date AND gov_revenue.freq = gdp.freq
""").df()
con.close()

# Merge Population & Sort
df = df.sort_values(['date'])
df = pd.merge(df, population[['date', 'population']], how='left', on='date')
df['population'] = df['population'].ffill()
df['revenue_per_capita'] = df['total'] / df['population']
df['date'] = pd.to_datetime(df['date'])

# ==========================================
# 2. PREPARE COLUMNS & GROUPING
# ==========================================
column_map = {
    'companies': 'direct_company',
    'petroleum_income_tax': 'direct_petroleum',
    'individual': 'direct_individual',
    'stamp_duty': 'direct_stamp_duties',
    'other_direct_taxes': 'direct_other',
    'sales_tax': 'indirect_sales_tax',
    'service_tax': 'indirect_services_tax',
    'export_duties': 'indirect_export_duties',
    'import_duties': 'indirect_import_duties',
    'excise_duties': 'indirect_excise_duties',
    'goods_and_services_tax': 'indirect_gs', 
    'other_indirect_taxes': 'indirect_other',
    'direct_tax': 'direct_total',
    'indirect_tax': 'indirect_total',
    'non_tax_revenue': 'non_tax_total',
}
df = df.rename(columns={k: v for k, v in column_map.items() if k in df.columns})

# Grouping Logic
df['direct_others_composite'] = df.get('direct_stamp_duties', 0) + df.get('direct_other', 0)
df['indirect_duties_composite'] = (
    df.get('indirect_export_duties', 0) + 
    df.get('indirect_import_duties', 0) + 
    df.get('indirect_excise_duties', 0)
)
df['indirect_others_composite'] = df.get('indirect_other', 0) + df.get('indirect_gs', 0)

if 'non_tax_total' not in df.columns:
    df['non_tax_total'] = (
        df.get('non_tax_roi', 0) + df.get('non_tax_other', 0) + 
        df.get('non_tax_licenses', 0) + df.get('non_tax_petroleum', 0)
    )

# ==========================================
# 3. APPLY GROWTH FUNCTION
# ==========================================
df_annual = df[df['freq'] == 'Annual'].copy()
df_quarterly = df[df['freq'] == 'Quarterly'].copy()
cols_to_exclude = [c for c in ['date', 'freq', 'year', 'quarter', 'type'] if c in df.columns]

df_annual_processed = derive_df_with_growth(df=df_annual, col_exclude=cols_to_exclude, yoy=1, periods_yoy=1, round_growth=2)
df_quarterly_processed = derive_df_with_growth(df=df_quarterly, col_exclude=cols_to_exclude, yoy=1, periods_yoy=4, round_growth=2)

# ==========================================
# 4. JSON GENERATOR (FINAL)
# ==========================================

def generate_json_from_processed(df_annual_proc, df_quarterly_proc):
    
    def to_timestamp(d): 
        if pd.isna(d): return None
        return int(d.timestamp() * 1000)

    def clean_for_json(series_or_list):
        data = series_or_list.tolist() if hasattr(series_or_list, 'tolist') else list(series_or_list)
        cleaned = []
        for x in data:
            if pd.isna(x) or x == np.inf or x == -np.inf:
                cleaned.append(None)
            else:
                cleaned.append(x)
        return cleaned

    # Metadata
    q_abs = df_quarterly_proc[df_quarterly_proc['series'] == 'abs']
    data_as_of = q_abs['date'].max().strftime('%Y-%m-%d %H:%M') if not q_abs.empty else "N/A"
    
    first_day_this_month = datetime.today().replace(day=1)
    last_day_last_month = first_day_this_month - timedelta(days=1)
    first_day_next_month = (first_day_this_month + timedelta(days=32)).replace(day=1)
    last_day_this_month = first_day_next_month - timedelta(days=1)
        
    output = {
        "metadata": {
            "data_as_of": data_as_of,
            "last_updated_at": "Last update: 2026-02-27",
            "next_update_at": "Next update: 2026-03-31",
        },
        "timeseries_main": {
            "data_as_of": data_as_of,
            "data": { "quarterly": {}, "yearly": {} }
        },
        "stack_bar_chart": {
            "data_as_of": data_as_of,
            "data": { 
                "quarterly": { "level": {}, "share_of_revenue": {}, "share_of_group": {} },
                "yearly": { "level": {}, "share_of_revenue": {}, "share_of_group": {} }
            }
        }
    }

    ts_columns = [
        'revenue_per_gdp', 'gov_deficit', 'revenue_per_capita',
        'direct_company', 'direct_individual', 'direct_petroleum', 'direct_stamp_duties', 'direct_other',
        'indirect_sales_tax', 'indirect_services_tax', 'indirect_export_duties', 
        'indirect_import_duties', 'indirect_excise_duties', 'indirect_other'
    ]
    
    sb_main = ['direct_total', 'indirect_total', 'non_tax_total', 'non_revenue_receipts']
    sb_direct = ['direct_company', 'direct_petroleum', 'direct_individual', 'direct_others_composite']
    sb_indirect = ['indirect_sales_tax', 'indirect_services_tax', 'indirect_duties_composite', 'indirect_others_composite']
    all_sb_cols = list(set(sb_main + sb_direct + sb_indirect))

    datasets = [('quarterly', df_quarterly_proc), ('yearly', df_annual_proc)]

    for key, dataset in datasets:
        if dataset.empty: continue
        
        # Alignment
        df_abs = dataset[dataset['series'] == 'abs'].sort_values('date')
        df_template = df_abs[['date']].copy()
        x_values = [to_timestamp(d) for d in df_template['date']]
        
        df_growth_raw = dataset[dataset['series'] == 'growth_yoy']
        df_growth = pd.merge(df_template, df_growth_raw, on='date', how='left')
        
        # --- Timeseries Section ---
        level_data = {'x': x_values}
        yoy_data = {'x': x_values}

        for col in ts_columns:
            if col in df_abs.columns:
                level_data[col] = clean_for_json(df_abs[col])
                yoy_data[col] = clean_for_json(df_growth[col])

        output['timeseries_main']['data'][key]['level'] = level_data
        output['timeseries_main']['data'][key]['yoy_growth'] = yoy_data

        # --- Stack Bar Section (Now applied for BOTH quarterly and yearly) ---
        sb_level = {'x': x_values}
        sb_share_rev = {'x': x_values}
        sb_share_group = {'x': x_values}
        
        group_totals = {
            'direct': df_abs.get('direct_total', df_abs['total']),
            'indirect': df_abs.get('indirect_total', df_abs['total']),
        }
        
        for col in all_sb_cols:
            if col in df_abs.columns:
                # Level
                sb_level[col] = clean_for_json(df_abs[col])
                
                # Share of Revenue
                total_rev = df_abs['total'].replace(0, np.nan)
                share_rev = (df_abs[col] / total_rev * 100)
                sb_share_rev[col] = clean_for_json(share_rev)
                
                # Share of Group
                if col in sb_direct: denom = group_totals['direct']
                elif col in sb_indirect: denom = group_totals['indirect']
                elif col in sb_main: denom = df_abs['total']
                else: denom = df_abs['total']
                
                denom = denom.replace(0, np.nan)
                share_group = (df_abs[col] / denom * 100)
                sb_share_group[col] = clean_for_json(share_group)

        # Store in nested structure
        output['stack_bar_chart']['data'][key]['level'] = sb_level
        output['stack_bar_chart']['data'][key]['share_of_revenue'] = sb_share_rev
        output['stack_bar_chart']['data'][key]['share_of_group'] = sb_share_group

    return output

# Execute
final_json = generate_json_from_processed(df_annual_processed, df_quarterly_processed)
with open('gov_revenue_new.json', 'w') as f:
    json.dump(final_json, f, indent=2)